# Exercise 3: RAG & Knowledge Integration

## Learning Objectives

In this exercise, you will:
- Understand the difference between tools (function calling) and RAG (knowledge retrieval)
- Configure and use Vertex AI RAG Engine for destination knowledge
- Create RAG-only agents with knowledge base access
- Test semantic retrieval with destination guides
- Combine function calling tools with RAG in a hybrid agent pattern

**Estimated time:** 40 minutes

## What is RAG (Retrieval-Augmented Generation)?

**⏱️ Estimated time: ~5 minutes**

<!-- INSTRUCTOR: This builds on Phase 2 - emphasize the complementary relationship -->

RAG gives your agent access to **private knowledge** - documents, guides, policies - that weren't in the LLM's training data. The agent retrieves relevant information from a knowledge base and uses it to generate accurate, grounded responses.

---

### How RAG Works

```
1. Documents indexed into RAG corpus (PDFs → chunks → embeddings)
          ↓
2. User asks: "What are visa requirements for Japan?"
          ↓
3. Agent searches corpus (semantic similarity)
          ↓
4. Top relevant chunks retrieved (with similarity scores)
          ↓
5. Agent uses chunks as context to answer question
```

---

### Tools vs RAG: The Decision Framework (Revisited)

Recall from Exercise 2 - you choose based on data characteristics:

```mermaid
flowchart TD
    A[User Query] --> B{Does this need\nREAL-TIME data?}
    B -->|YES| C[Use FUNCTION CALLING\nsearch_flights, search_hotels]
    B -->|NO| D{Is this STATIC\nKNOWLEDGE?}
    D -->|YES| E[Use RAG RETRIEVAL\nretrieve_destination_info]
    D -->|NO| F[Direct LLM\ngeneral knowledge]
```

| Question | Tool or RAG? | Why? |
|----------|--------------|------|
| "Find flights to Tokyo" | **TOOL** (search_flights) | Prices/availability change constantly |
| "What's the best time to visit Tokyo?" | **RAG** (retrieve_destination_info) | Static seasonal guide |
| "Book hotel in Shibuya under $200" | **TOOL** (search_hotels) | Real-time pricing |
| "What are visa requirements for Japan?" | **RAG** (retrieve_destination_info) | Static immigration policy |
| "What's the capital of Japan?" | **Neither** (LLM knows this) | General knowledge |

> 💡 **Key Insight:** Use **tools for dynamic data** that changes while user is talking. Use **RAG for static knowledge** in documents.

---

### Why RAG for Travel Guides?

**Perfect use case:**
- Destination guides rarely change (update quarterly, not hourly)
- Contains structured information (visa rules, attractions, cultural tips)
- Too much content to fit in LLM context window
- Needs semantic search ("safety tips" retrieves safety sections)

**In this workshop:**
- Pre-indexed corpus: 10 destination guides (Tokyo, Paris, NYC, Singapore, etc.)
- Each guide: visa requirements, attractions, weather, culture, transportation, food
- Layout-aware chunking: tables and lists preserved intact
- Document AI parser: understands document structure for better retrieval

## Setup Instructions

**⏱️ Estimated time: ~2 minutes**

This exercise assumes you have completed Exercise 1 and Exercise 2.

### ⚠️ Important: Different Authentication Required

**Exercises 1-2** used a Google AI API key (simple, works everywhere).

**Exercise 3** uses **Vertex AI RAG Engine**, which requires:
1. **Google Cloud authentication** (sign in with your Google account)
2. **Access to the workshop project** (your instructor has granted this)

When you run the setup cell, a popup will ask you to sign in. This grants Colab access to query the RAG corpus.

### Why the difference?

| Feature | Google AI API | Vertex AI |
|---------|---------------|-----------|
| Authentication | API key | Google Cloud IAM |
| RAG Engine | ❌ Not supported | ✅ Supported |
| Best for | Quick prototypes | Production, enterprise |

If you haven't completed setup yet, please go back to: [00-setup-verification.ipynb](00-setup-verification.ipynb)

In [ ]:
# @title Quick Setup (Run this cell first)
# This cell sets up your environment - you can collapse it after running

# ⚠️ IMPORTANT: Exercise 3 uses Vertex AI RAG Engine, which requires
# Google Cloud authentication (not just an API key like Exercises 1-2)

import os

# Step 1: Install Google ADK
!pip install -q google-adk==1.23.0 google-cloud-aiplatform
print("✓ google-adk and google-cloud-aiplatform installed")

# Step 2: Authenticate with Google Cloud
# This opens a popup to sign in with your Google account
from google.colab import auth
auth.authenticate_user()
print("✓ Authenticated with Google Cloud")

# Step 3: Configure ADK to use Vertex AI (CRITICAL!)
# This tells ADK to use Vertex AI backend instead of Google AI API
VERTEX_PROJECT = "674082857580"  # Workshop project with RAG corpus
VERTEX_LOCATION = "europe-west1"

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'  # <-- This is the key!
os.environ['GOOGLE_CLOUD_PROJECT'] = VERTEX_PROJECT
os.environ['GOOGLE_CLOUD_LOCATION'] = VERTEX_LOCATION

# Remove any API key that might interfere
if 'GOOGLE_API_KEY' in os.environ:
    del os.environ['GOOGLE_API_KEY']

# Initialize Vertex AI
import vertexai
vertexai.init(project=VERTEX_PROJECT, location=VERTEX_LOCATION)

print(f"✓ Vertex AI initialized")
print(f"  Project: {VERTEX_PROJECT}")
print(f"  Location: {VERTEX_LOCATION}")
print(f"  GOOGLE_GENAI_USE_VERTEXAI: {os.environ.get('GOOGLE_GENAI_USE_VERTEXAI')}")

print("\n🚀 Ready to integrate RAG knowledge!")
print("\n⚠️ Note: Unlike Exercises 1-2, RAG requires Vertex AI (not API keys)")

In [ ]:
# Import ADK components for agents and RAG
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part
from google.adk.tools.retrieval.vertex_ai_rag_retrieval import VertexAiRagRetrieval
from vertexai.preview import rag

# Create session service for conversation management
session_service = InMemorySessionService()
user_id = "workshop_participant"

print("✓ ADK components imported")
print("✓ RAG retrieval tools imported")
print("✓ Session service created")
print("✓ Using Vertex AI backend (GOOGLE_GENAI_USE_VERTEXAI=TRUE)")

## Exercise 3A: Explore Pre-Indexed RAG Corpus

**⏱️ Estimated time: ~5 minutes**

<!-- INSTRUCTOR: Show the corpus in the console if possible -->

The instructor has pre-created a RAG corpus with destination guide PDFs. This saves 15-20 minutes of setup time and lets us focus on **using** RAG, not infrastructure.

### What's in the Corpus?

**10 destination guides**, each with standardized sections:
1. Quick Facts (language, currency, emergency numbers)
2. Visa Requirements (by nationality)
3. Best Time to Visit (seasonal breakdown)
4. Top Attractions (table format with fees, hours, tips)
5. Neighborhoods & Districts
6. Food & Dining
7. Transportation
8. Cultural Tips & Customs
9. Safety & Health
10. Practical Information

**Destinations covered:**
- Tokyo, Japan
- Paris, France
- New York City, USA
- Singapore
- London, UK
- Rome, Italy
- Bangkok, Thailand
- Sydney, Australia
- Barcelona, Spain
- Dubai, UAE

### How the Guides Were Indexed

**Chunking strategy:**
- Chunk size: 1024 tokens (~800 words)
- Chunk overlap: 256 tokens (~200 words)
- Document AI layout parser enabled (preserves tables/lists)

**Why layout parsing matters:**
- WITHOUT: Table splits mid-row → broken data
- WITH: Full tables in single chunk → clean retrieval

Example:
```
| Attraction | Price | Hours |
|------------|-------|-------|
| Temple     | Free  | 6am   |
| Museum     | $15   | 9am   |
```
↑ This stays intact in one chunk with layout parser

In [ ]:
# Configure RAG Corpus ID
# The instructor will provide this during the workshop

# Workshop corpus (pre-indexed with 10 destination guides)
RAG_CORPUS_ID = "projects/674082857580/locations/europe-west1/ragCorpora/7493989779944505344"

# Store in environment for later use
os.environ['RAG_CORPUS_ID'] = RAG_CORPUS_ID

print(f"✓ RAG Corpus ID configured")
print(f"  Corpus: {RAG_CORPUS_ID[:60]}...")
print("\n📚 Corpus contains 10 destination guides with ~15-25 chunks each")

### ✅ Checkpoint: Corpus Configured

<!-- INSTRUCTOR: Verify all participants have the corpus ID before proceeding -->

You should see:
- ✓ RAG Corpus ID configured
- Corpus path starting with `projects/...`

**If you see an error:**
- Make sure you replaced the placeholder corpus ID with the one from your instructor
- Check that the format matches: `projects/{project}/locations/{location}/ragCorpora/{id}`

---

## ✏️ Exercise 3B: Configure RAG Retrieval Tool

**⏱️ Estimated time: ~7 minutes**

<!-- INSTRUCTOR: Emphasize the DO/DO NOT pattern - critical for tool selection -->

Now let's create a RAG retrieval tool that searches the destination guide corpus.

### VertexAiRagRetrieval Parameters

| Parameter | Type | Purpose | Recommended Value |
|-----------|------|---------|-------------------|
| `name` | str | Function name LLM calls | `retrieve_destination_info` |
| `description` | str | **Critical!** Tells LLM when to use this tool | See pattern below |
| `rag_resources` | list | Corpus reference(s) to search | `[rag.RagResource(rag_corpus=...)]` |
| `similarity_top_k` | int | How many chunks to retrieve | `5` (default) |
| `vector_distance_threshold` | float | Minimum similarity score (0-1) | `0.6` (filters low quality) |

### The DO/DO NOT Description Pattern

**Why this matters:** The LLM reads your tool description to decide when to call it. Vague descriptions cause tool confusion!

❌ **Bad description:**
```python
description='Get destination information'  # Too vague!
```
Result: LLM calls this for "Find flights to Tokyo" 😞

✅ **Good description:**
```python
description='''
Retrieve destination information from travel guide knowledge base.

USE THIS TOOL to answer questions about:
- Visa requirements and entry rules
- Top attractions and landmarks  
- Weather and best time to visit
- Cultural tips and customs

DO NOT use this tool for:
- Real-time flight availability → use search_flights() instead
- Real-time hotel availability → use search_hotels() instead
'''
```
Result: LLM calls RAG for guides, tools for bookings 🎯

### Your Task

Complete the `VertexAiRagRetrieval` configuration below:

In [ ]:
# Exercise 3B: Configure RAG Tool

# Define retrieval parameters (we'll use these values when creating the tool)
SIMILARITY_TOP_K = 5      # How many chunks to retrieve
VECTOR_THRESHOLD = 0.6    # Minimum similarity score (0-1)

destination_knowledge = VertexAiRagRetrieval(
    name='retrieve_destination_info',
    
    # TODO: Write explicit description with DO/DO NOT sections
    # Hint: Explain what information this tool provides
    # Hint: Clarify when to use this vs booking tools (search_flights, search_hotels)
    description='''
    TODO: Fill in description
    
    USE THIS TOOL to answer questions about:
    - TODO: List what RAG provides (visa, attractions, culture...)
    
    DO NOT use this tool for:
    - TODO: List what tools provide (real-time flights, hotels...)
    ''',
    
    # TODO: Configure RAG resources with corpus ID
    rag_resources=[
        rag.RagResource(
            rag_corpus=os.environ['RAG_CORPUS_ID']  # Use the environment variable
        )
    ],
    
    # TODO: Set retrieval parameters
    similarity_top_k=SIMILARITY_TOP_K,  # How many chunks to retrieve? (try 5)
    vector_distance_threshold=VECTOR_THRESHOLD,  # Minimum similarity? (try 0.6)
)

print("✓ RAG tool configured")
print(f"  Retrieval: top {SIMILARITY_TOP_K} chunks")
print(f"  Threshold: {VECTOR_THRESHOLD} similarity")

### 💡 Solution: Complete Configuration

Click to expand if you need help:

In [ ]:
# @title Solution: Exercise 3B (Expand to see)

destination_knowledge_solution = VertexAiRagRetrieval(
    name='retrieve_destination_info',
    
    description='''Retrieve destination information from travel guide knowledge base.

USE THIS TOOL to answer questions about:
- Visa requirements and entry rules (static immigration policy)
- Top attractions and landmarks (guide recommendations)
- Best time to visit by season (weather patterns, events)
- Cultural tips and local customs (etiquette, traditions)
- Safety information and travel advisories
- Transportation within the city (metro, buses, taxis)
- Food and dining recommendations (local cuisine, prices)
- Neighborhoods and districts to explore
- Practical travel information (plugs, currency, language)

DO NOT use this tool for:
- Real-time flight availability → use search_flights() instead
- Real-time hotel availability → use search_hotels() instead
- Current pricing or booking status → use search tools
- Live event schedules or ticket availability

This tool searches STATIC destination guides, not live databases.
Use for travel planning knowledge, not booking transactions.''',
    
    rag_resources=[
        rag.RagResource(
            rag_corpus=os.environ['RAG_CORPUS_ID']
        )
    ],
    
    similarity_top_k=5,
    vector_distance_threshold=0.6,
)

print("✓ Solution RAG tool configured")
print("  Retrieval: top 5 chunks, threshold 0.6")

### ✅ Checkpoint: Tool Configured

You should see:
- ✓ RAG tool configured
- Retrieval: top 5 chunks
- Threshold: 0.6 similarity

**If you get an error:**
- Check that `RAG_CORPUS_ID` is set (from Exercise 3A)
- Verify the description is a multi-line string (use triple quotes `'''`)
- Make sure all parameters are filled in

---

## ✏️ Exercise 3C: Create RAG-Only Agent

**⏱️ Estimated time: ~5 minutes**

<!-- INSTRUCTOR: Explain the single-tool constraint clearly -->

Now create an agent that uses **ONLY** the RAG retrieval tool for destination knowledge.

### ⚠️ Important ADK Constraint

**VertexAiRagRetrieval can ONLY be used by itself** - you cannot combine it with function calling tools in the same agent.

❌ **This will NOT work:**
```python
agent = Agent(
    tools=[
        search_flights,        # Function calling tool
        search_hotels,         # Function calling tool  
        destination_knowledge, # RAG tool ← VIOLATION!
    ]
)
```

✅ **This WILL work:**
```python
# RAG-only agent
destination_agent = Agent(
    tools=[destination_knowledge]  # ONLY RAG tool
)
```

**Why this constraint?** ADK's architecture limitation. We'll solve this in Exercise 3E with a "hybrid agent" pattern.

### Your Task

Complete the agent configuration:

In [ ]:
# Exercise 3C: Create RAG-Only Agent

destination_expert = Agent(
    model='gemini-2.5-flash',
    name='destination_expert',
    
    # TODO: Write agent description
    description='TODO: Describe this agent (e.g., "Travel destination expert with knowledge base access")',
    
    # TODO: Write agent instruction
    instruction='''TODO: Write instruction for the agent
    
    Your instruction should explain:
    - What the agent can do (retrieve from destination guides)
    - What it cannot do (real-time booking)
    - How to use the retrieve_destination_info tool
    - When to cite information from guides
    
    Example structure:
    "You are a travel destination expert with access to comprehensive guides.
    
    YOUR CAPABILITIES:
    - Retrieve information from destination guides using your tool
    - Answer questions about visa requirements, attractions, weather, culture
    ...
    
    YOUR LIMITATIONS:
    - You CANNOT search for flights or hotels (no real-time booking data)
    ...
    
    HOW TO HELP:
    1. Use retrieve_destination_info tool for all destination queries
    2. Combine information from multiple guide sections
    3. Cite specific details when available
    ..."
    ''',
    
    # TODO: Add ONLY the RAG tool (single-tool constraint!)
    tools=[destination_knowledge],  # ONLY this tool, no others!
)

print("✓ Destination expert agent created")
print(f"  Model: {destination_expert.model}")
print(f"  Tools: {len(destination_expert.tools)} (RAG only)")

### 💡 Solution: Complete Agent Configuration

Click to expand if you need help:

In [ ]:
# @title Solution: Exercise 3C (Expand to see)

destination_expert_solution = Agent(
    model='gemini-2.5-flash',
    name='destination_expert',
    description='Travel destination expert with access to comprehensive destination guides.',
    
    instruction='''You are a travel destination expert with access to comprehensive destination guides.

YOUR CAPABILITIES:
- Retrieve information from destination guides using your tool
- Answer questions about visa requirements, attractions, weather, culture
- Provide cultural tips, safety info, transportation advice
- Share food and dining recommendations

YOUR LIMITATIONS:
- You CANNOT search for flights or hotels (no real-time booking data)
- You CANNOT provide current prices or booking availability
- Your knowledge comes from static travel guides

HOW TO HELP:
1. Use retrieve_destination_info tool for all destination queries
2. Combine information from multiple guide sections for comprehensive answers
3. Cite specific details from guides when available
4. If guide doesn't cover a topic, admit this honestly
5. Recommend using booking tools for real-time availability

Be informative, encouraging, and help travelers prepare confidently.''',
    
    tools=[destination_knowledge_solution],
)

print("✓ Solution destination expert created")

### ✅ Checkpoint: Agent Created

You should see:
- ✓ Destination expert agent created
- Model: gemini-2.5-flash
- Tools: 1 (RAG only)

**Common issues:**
- If "Tools: 0" → You forgot to add `destination_knowledge` to the tools list
- If "Tools: 2+" → You added other tools - remove them! RAG must be alone

---

## ✏️ Exercise 3D: Test RAG Retrieval

**⏱️ Estimated time: ~8 minutes**

<!-- INSTRUCTOR: Walk through the first test query together -->

Let's test our RAG-only agent with destination queries!

### What to Watch For

When you run a test query, observe:
1. **Tool call**: Does the agent call `retrieve_destination_info`?
2. **Retrieved chunks**: Are they relevant to the question?
3. **Response quality**: Does the answer cite guide information?
4. **Similarity scores**: (internal) How confident is retrieval?

### Test Queries

We'll test queries covering different destinations and guide sections:

In [ ]:
# Helper function to test agent queries
async def test_destination_query(agent, query):
    """Run a single query against the destination expert agent."""
    # Create session
    session = await session_service.create_session(
        app_name=agent.name,
        user_id=user_id
    )
    
    # Create runner
    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )
    
    print(f"\n{'='*60}")
    print(f"🧑 You: {query}")
    print(f"{'='*60}")
    
    final_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(parts=[Part(text=query)], role="user")
    ):
        # Debug: print event type
        # print(f"  [DEBUG] Event: {type(event).__name__}, is_final={event.is_final_response()}")
        
        # Watch for RAG tool calls
        if event.content and hasattr(event.content, 'parts'):
            for part in event.content.parts:
                if hasattr(part, 'function_call') and part.function_call:
                    print(f"🔧 Tool called: {part.function_call.name}")
        
        if event.is_final_response():
            # Handle different response structures
            if event.content and hasattr(event.content, 'parts') and event.content.parts:
                final_response = event.content.parts[0].text
            elif hasattr(event, 'text'):
                final_response = event.text
            else:
                # Debug: show what we got
                print(f"  [DEBUG] Event attributes: {dir(event)}")
                if event.content:
                    print(f"  [DEBUG] Content: {event.content}")
                final_response = "(No text response found)"
            break
    
    print(f"\n🤖 Agent: {final_response}")
    print("-" * 60)

In [ ]:
# Test 1: Visa requirements (Tokyo guide)
await test_destination_query(
    destination_expert,
    "What are the visa requirements for US citizens traveling to Japan?"
)

# Expected:
# - Tool call: retrieve_destination_info
# - Answer should mention visa-free entry, 90 days, etc.
# - Should cite specific details from Tokyo guide

In [ ]:
# Test 2: Best time to visit (Paris guide)
await test_destination_query(
    destination_expert,
    "What's the best time to visit Paris?"
)

# Expected:
# - Tool call: retrieve_destination_info
# - Answer should discuss seasons, weather, events
# - May mention spring for gardens, avoiding summer crowds

In [ ]:
# Test 3: Cultural customs (Tokyo guide)
await test_destination_query(
    destination_expert,
    "What cultural customs should I know before visiting Tokyo?"
)

# Expected:
# - Tool call: retrieve_destination_info
# - Answer should cover etiquette (bowing, shoes, chopsticks, etc.)
# - Should be specific to Japanese culture

In [ ]:
# Test 4: Attractions (New York guide)
await test_destination_query(
    destination_expert,
    "What are the top attractions in New York City?"
)

# Expected:
# - Tool call: retrieve_destination_info
# - Answer should list landmarks (Statue of Liberty, Central Park, etc.)
# - May include practical info (entry fees, hours)

### ✅ Checkpoint: RAG Retrieval Working

<!-- INSTRUCTOR: Review outputs with participants, check quality -->

You should see:
- 🔧 Tool called: `retrieve_destination_info` for each query
- Relevant answers citing destination guide information
- Specific details (visa rules, weather patterns, customs)

**Common issues:**
- **No tool call**: Check that agent has `destination_knowledge` in tools list
- **Generic answers**: Agent might not be retrieving chunks - verify `RAG_CORPUS_ID`
- **Wrong destination**: Retrieval working but query ambiguous - try more specific queries

**What just happened?**
1. Your query → Agent decides to use RAG tool
2. Query converted to embedding vector
3. Semantic search in corpus (cosine similarity)
4. Top 5 most relevant chunks retrieved (threshold 0.6)
5. Chunks used as context for LLM to generate answer
6. Answer grounded in guide content (not hallucinated!)

---

## ✏️ Exercise 3E: Hybrid Agent Pattern

**⏱️ Estimated time: ~10 minutes**

<!-- INSTRUCTOR: This is the key pattern - explain the architecture clearly -->

Now for the real power: **combining function calling tools with RAG knowledge**.

Recall the constraint: VertexAiRagRetrieval cannot mix with other tools in the same agent.

### The Solution: Sequential Agent Coordination

```
┌─────────────────────────────────────┐
│   User Query                        │
│   "Find flights to Tokyo + visa"    │
└────────────┬────────────────────────┘
             │
             ↓
┌─────────────────────────────────────┐
│   COORDINATOR                       │
│   (Intent Detection & Routing)      │
└────┬─────────────────────┬──────────┘
     │                     │
     ↓                     ↓
┌─────────────┐      ┌──────────────┐
│ Booking     │      │ Destination  │
│ Agent       │      │ Agent        │
│             │      │              │
│ Tools:      │      │ Tools:       │
│ - flights   │      │ - RAG only   │
│ - hotels    │      │              │
└─────────────┘      └──────────────┘
     │                     │
     └──────────┬──────────┘
                ↓
      ┌─────────────────┐
      │ Combined Result │
      │ Flights + Tips  │
      └─────────────────┘
```

### How It Works

1. **Intent Detection**: Analyze query for booking vs knowledge keywords
2. **Route to Agent**: 
   - Booking keywords → booking agent (search tools)
   - Knowledge keywords → destination agent (RAG tool)
   - Both → booking agent + enrich with RAG tips
3. **Combine Results**: Return comprehensive response

### Your Task

Implement a simple hybrid coordinator:

In [ ]:
# First, we need the booking tools from Exercise 2
# Copy your search_flights and search_hotels functions here
# (Or import from reference implementation)

from datetime import datetime
from typing import Optional

def search_flights(
    origin: str,
    destination: str,
    departure_date: str,
    passengers: int = 1,
    max_price: Optional[int] = None
) -> dict:
    """Search for available flights (mock data)."""
    print(f"🔧 search_flights called: {origin} -> {destination}")
    
    # Validate date
    try:
        datetime.strptime(departure_date, '%Y-%m-%d')
    except ValueError:
        return {
            "status": "error",
            "error_message": f"Invalid date format: '{departure_date}'. Use YYYY-MM-DD."
        }
    
    # Mock flight data
    all_flights = [
        {"airline": "United", "price": 850, "departure": "08:30"},
        {"airline": "ANA", "price": 920, "departure": "11:00"},
        {"airline": "JAL", "price": 890, "departure": "13:45"},
    ]
    
    # Filter by budget
    if max_price:
        flights = [f for f in all_flights if f["price"] <= max_price]
    else:
        flights = all_flights
    
    return {
        "status": "success",
        "flights": flights,
        "route": f"{origin} → {destination}",
        "date": departure_date,
    }

def search_hotels(
    location: str,
    check_in: str,
    check_out: str,
    guests: int = 1,
    max_price_per_night: Optional[int] = None
) -> dict:
    """Search for available hotels (mock data)."""
    print(f"🔧 search_hotels called: {location}")
    
    # Mock hotel data
    all_hotels = [
        {"name": "Park Hyatt Tokyo", "price_per_night": 450},
        {"name": "Shinjuku Granbell", "price_per_night": 180},
        {"name": "MUJI Hotel Ginza", "price_per_night": 220},
    ]
    
    # Filter by budget
    if max_price_per_night:
        hotels = [h for h in all_hotels if h["price_per_night"] <= max_price_per_night]
    else:
        hotels = all_hotels
    
    return {
        "status": "success",
        "hotels": hotels,
        "location": location,
    }

print("✓ Booking tools defined")

In [ ]:
# Create booking agent (tools only)
booking_agent = Agent(
    model='gemini-2.5-flash',
    name='booking_agent',
    description='Search for flights and hotels with real-time availability.',
    instruction='''You search for travel bookings using real-time data.

YOUR CAPABILITIES:
- Search for flights between airports
- Search for hotels in destinations
- Filter by budget constraints

HOW TO HELP:
1. Use search_flights() for flight queries
2. Use search_hotels() for accommodation queries
3. Apply max_price when budget mentioned
4. Present 2-3 best options

You CANNOT provide destination information (visa, culture, weather).
For that, recommend consulting destination guides.''',
    tools=[search_flights, search_hotels],
)

print("✓ Booking agent created")
print(f"  Tools: {len(booking_agent.tools)} (search_flights, search_hotels)")

In [ ]:
# TODO: Implement hybrid coordinator function

async def hybrid_travel_assistant(user_query: str) -> str:
    """
    Hybrid travel assistant combining booking and destination agents.
    
    Routes queries to appropriate agent based on intent.
    Enriches booking results with destination tips.
    """
    query_lower = user_query.lower()
    
    # TODO: Define intent detection keywords
    booking_keywords = ['flight', 'hotel', 'book', 'search', 'find', 'price']
    knowledge_keywords = ['visa', 'weather', 'attraction', 'culture', 'custom', 'tip']
    
    # Detect intent
    needs_booking = any(kw in query_lower for kw in booking_keywords)
    needs_knowledge = any(kw in query_lower for kw in knowledge_keywords)
    
    # Helper to run agent
    async def run_agent(agent, query):
        session = await session_service.create_session(
            app_name=agent.name,
            user_id='hybrid_user'
        )
        runner = Runner(
            agent=agent,
            session_service=session_service,
            app_name=agent.name
        )
        async for event in runner.run_async(
            user_id='hybrid_user',
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if event.is_final_response():
                # Handle different response structures (fix for None content)
                if event.content and hasattr(event.content, 'parts') and event.content.parts:
                    return event.content.parts[0].text
                elif hasattr(event, 'text') and event.text:
                    return event.text
                else:
                    return "(No response received)"
        return "(No final response)"
    
    # TODO: Route based on intent
    if needs_knowledge and not needs_booking:
        # Knowledge-only query → destination agent
        print("📚 Routing to destination agent...")
        return await run_agent(destination_expert, user_query)
    
    elif needs_booking:
        # Booking query → booking agent
        print("🔧 Routing to booking agent...")
        booking_result = await run_agent(booking_agent, user_query)
        
        # TODO: Optionally enrich with destination tips
        # Extract destination from query (simplified)
        destinations = {
            'tokyo': 'Tokyo', 'japan': 'Tokyo',
            'paris': 'Paris', 'france': 'Paris',
            'new york': 'New York', 'nyc': 'New York',
        }
        
        destination = None
        for keyword, city in destinations.items():
            if keyword in query_lower:
                destination = city
                break
        
        # Enrich if destination detected
        if destination:
            print(f"📚 Enriching with {destination} tips...")
            tip_query = f"What are the top 3 tips for visiting {destination}?"
            tips = await run_agent(destination_expert, tip_query)
            return f"{booking_result}\n\n**{destination} Travel Tips:**\n{tips}"
        
        return booking_result
    
    else:
        # Default to booking agent
        print("🔧 Routing to booking agent (default)...")
        return await run_agent(booking_agent, user_query)

print("✓ Hybrid assistant function defined")

In [ ]:
# Test 1: Booking with enrichment
print("\n" + "="*60)
print("Test 1: Booking Query with Destination")
print("="*60)

response = await hybrid_travel_assistant(
    "Find me flights from SFO to Tokyo on March 15, 2026. Budget $900."
)
print(response)

In [ ]:
# Test 2: Knowledge-only query
print("\n" + "="*60)
print("Test 2: Knowledge Query")
print("="*60)

response = await hybrid_travel_assistant(
    "What cultural customs should I know before visiting Tokyo?"
)
print(response)

In [ ]:
# Test 3: Mixed query (both booking and knowledge)
print("\n" + "="*60)
print("Test 3: Mixed Query")
print("="*60)

response = await hybrid_travel_assistant(
    "I want to book a hotel in Paris for March 20-25, and I'd like to know about the weather."
)
print(response)

### ✅ Checkpoint: Hybrid Pattern Working

You should see:
- **Test 1**: Flight results + Tokyo travel tips (both agents called)
- **Test 2**: Cultural customs answer (destination agent only)
- **Test 3**: Hotel results + Paris weather info (both agents called)

**What just happened?**
- Intent detection routed queries to appropriate agent(s)
- Booking queries used function calling tools
- Knowledge queries used RAG retrieval
- Mixed queries combined both!

**This pattern scales:**
- Production: Replace keyword matching with NER/classification
- Add more specialized agents (visa agent, transport agent)
- Use orchestration frameworks (LangGraph) for complex routing

---

## Challenge (Optional)

Ready for more? Try these challenges:

### Challenge 1: Tune Retrieval Quality

Experiment with `similarity_top_k` and `vector_distance_threshold`:
- Create a new RAG tool with `top_k=10` and `threshold=0.4`
- Test with same queries from Exercise 3D
- Compare answer quality: More chunks = better context or more noise?

### Challenge 2: Multi-Destination Query

Test: "Compare visa requirements for US citizens visiting Japan vs France"
- Does retrieval find both Tokyo and Paris guides?
- Does answer compare the two?

### Challenge 3: Edge Case Handling

Test: "What are visa requirements for Mars?"
- Does the agent return low-similarity results or say "no information found"?
- This tests your `vector_distance_threshold` filtering

### Challenge 4: Improve Intent Detection

Enhance the `hybrid_travel_assistant` function:
- Add more destination keywords (Singapore, London, Rome)
- Handle queries like "flights and visa" (clearly both)
- Add logging to show routing decisions

In [ ]:
# Your challenge code here
# TODO: Experiment with retrieval parameters
# TODO: Test edge cases
# TODO: Improve coordinator logic

---

## What You Learned

🎉 **Congratulations!** You've mastered RAG integration with ADK!

### Key Concepts

✅ **RAG vs Tools**: Static knowledge (RAG) vs real-time data (tools)  
✅ **Vertex AI RAG Engine**: Managed corpus, semantic search, layout-aware chunking  
✅ **Tool Descriptions**: DO/DO NOT pattern for correct LLM tool selection  
✅ **Single-Tool Constraint**: VertexAiRagRetrieval must be alone in agent  
✅ **Hybrid Pattern**: Sequential coordination with specialized agents  
✅ **Intent Detection**: Route queries to booking or knowledge agents  
✅ **Result Enrichment**: Combine tool outputs with RAG insights

### Architecture Patterns You Built

1. **RAG-Only Agent**: Destination expert with knowledge base access
2. **Function-Only Agent**: Booking agent with real-time search tools
3. **Hybrid Coordinator**: Routes queries and combines results

### Production Considerations

**What we used (workshop):**
- Keyword-based intent detection
- Simple destination extraction (dictionary lookup)
- Sequential agent execution

**What production needs:**
- NER or classification model for intent
- Entity recognition for destinations
- Caching RAG results for repeated queries
- Monitoring retrieval quality metrics
- Orchestration framework for complex workflows

### What's Next?

**Coming up in Exercise 4:** Session management and conversation memory
- Multi-turn conversations
- Context preservation across queries
- Conversation history management

**Advanced RAG topics** (bonus content):
- Creating your own destination guide and adding to corpus
- Chunking strategy optimization
- Multiple corpora (separate guides from policies)
- RAG evaluation metrics (faithfulness, relevance)

**Resources:**
- [Vertex AI RAG Engine Docs](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-overview)
- [ADK RAG Tool Reference](https://google.github.io/adk-docs/tools/google-cloud/vertex-ai-rag-engine/)
- [Document AI Layout Parser](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/layout-parser-integration)

Ready to continue? Open the next exercise notebook!